In [ ]:
!nvidia-smi
%matplotlib inline

In [1]:
import os
import pandas as pd
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from tqdm import tqdm
from typing import List, Literal, List, Dict, Any, Optional
import numpy as np
import seaborn as sns

from datasets import load_dataset
import random
import json
import re
from functools import partial
from datasets import Dataset
from copy import deepcopy
import evaluate
import nltk
from scipy.stats import ttest_ind
import string
from collections import Counter

import openai
import os
import time
import pandas as pd
import torch

from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from dotenv import load_dotenv
load_dotenv()

/home/yhuang/ondemand/paper_clean/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Data Preprocessing

In [5]:
frames_benchmark = load_dataset("google/frames-benchmark")['test']

README.md: 0.00B [00:00, ?B/s]

test.tsv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/824 [00:00<?, ? examples/s]

In [6]:
frames_benchmark[0]

{'Unnamed: 0': 0,
 'Prompt': "If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name? ",
 'Answer': 'Jane Ballou',
 'wikipedia_link_1': 'https://en.wikipedia.org/wiki/President_of_the_United_States',
 'wikipedia_link_2': 'https://en.wikipedia.org/wiki/James_Buchanan',
 'wikipedia_link_3': 'https://en.wikipedia.org/wiki/Harriet_Lane',
 'wikipedia_link_4': 'https://en.wikipedia.org/wiki/List_of_presidents_of_the_United_States_who_died_in_office',
 'wikipedia_link_5': 'https://en.wikipedia.org/wiki/James_A._Garfield',
 'wikipedia_link_6': None,
 'wikipedia_link_7': None,
 'wikipedia_link_8': None,
 'wikipedia_link_9': None,
 'wikipedia_link_10': None,
 'wikipedia_link_11+': None,
 'reasoning_types': 'Multiple constraints',
 'wiki_links': "['https://en.wikipedia.org/wiki/President_of_the_United_States', 'https://en.wikipedia.org/wiki/

In [7]:
def simplify(example):
    return {
        "id": example["Unnamed: 0"],
        "request": example["Prompt"],
        "answer": example["Answer"]
    }

simplified_frames_benchmark = frames_benchmark.map(simplify, remove_columns=frames_benchmark.column_names)
print(simplified_frames_benchmark[0])

Map:   0%|          | 0/824 [00:00<?, ? examples/s]

{'id': 0, 'request': "If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name? ", 'answer': 'Jane Ballou'}


In [8]:
simplified_frames_benchmark.to_json("frames_sample_raw.jsonl",orient="records", lines=True)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

183450

## Using off-the-shelf Qwen3-4B for classification

In [2]:
import os
print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE"))

#os.environ["HF_HUB_DISABLE_XET"] = "1"
#os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

# 可选：确保没被错误地关掉进度条
# os.environ.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)
#os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

HF_HOME: /scratch-local/yhuang/huggingface_cache
HF_DATASETS_CACHE: /scratch-local/yhuang/huggingface_cache/datasets
HF_HUB_CACHE: /scratch-local/yhuang/huggingface_cache/hub


In [3]:
frames_sample = load_dataset(
    "json",
    data_files="frames_sample_raw.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

Generating train split: 824 examples [00:00, 132773.97 examples/s]


In [4]:
from helper_functions_qa import (prepare_test_prompts, run_experiment, merge_df_into_dataset_by_order)

In [5]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [6]:
Qwen3_4B = "Qwen/Qwen3-4B"

# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)

# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.42s/it]


cuda


In [7]:
df = pd.read_json("frames_sample_raw.jsonl", lines=True)
test_prompts = prepare_test_prompts(df, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 824
Generation complete: 824 prompts
Average prompt length: 531 bytes (~132 tokens)

Analyze the following input user query:

{"query": "If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name? "}

Please provide your analysis in the following JSON format:

{"query": "If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name? ", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [8]:
test_df = run_experiment(tokenizer, model, test_prompts, system_prompt, df)
test_df.to_csv('./intermediate/BASELINE_frames_classified.csv')
test_df

100%|██████████| 165/165 [3:19:41<00:00, 72.62s/it]   


,id,request,answer,thinking,model_response,model_pred
0,0,If my future wife has the same first name as t...,Jane Ballou,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""If my future wife has the sam...",fully specified
1,1,Imagine there is a building called Bronte towe...,37th,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""Imagine there is a building cal...",fully specified
2,2,How many years earlier would Punxsutawney Phil...,87,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""How many years earlier would ...",underspecified
3,3,"As of August 1, 2024, which country were holde...",France,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""As of August 1, 2024, which cou...",underspecified
4,4,What is the name of the vocalist from the firs...,Jens Kidman,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What is the name of the vocal...",underspecified
...,...,...,...,...,...,...
819,819,How many years after publishing his paper *On ...,Seven,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""How many years after publishing...",fully specified
820,820,This singer represented Sweden in Eurovision f...,Aquarius,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""This singer represented Sweden ...",fully specified
821,821,Who was the king of England when Isaac Newton ...,James II of England,"<think>\nOkay, let's see. The user is asking, ...","{\n ""query"": ""Who was the king of England whe...",fully specified
822,822,Which movie musical produced a song that was i...,Fame,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""Which movie musical produced a ...",underspecified


In [9]:
annotated_dataset = merge_df_into_dataset_by_order(
    frames_sample, 
    test_df, 
    columns=["thinking", "model_response", "model_pred"],
    prefix="qwen3_",
    inplace=False
)

In [10]:
# Create the UND subset
underspecified_set = annotated_dataset.filter(
    lambda x: x["qwen3_model_pred"].strip().lower() == "underspecified"
)

# Create the FS subset
fully_specified_set = annotated_dataset.filter(
    lambda x: x["qwen3_model_pred"].strip().lower() == "fully specified"
)

# The size of subsets
print(f"Underspecified samples: {len(underspecified_set)}")
print(f"Fully specified samples: {len(fully_specified_set)}")

Filter: 100%|██████████| 824/824 [00:00<00:00, 69310.65 examples/s]

Underspecified samples: 440
Fully specified samples: 384


In [11]:
annotated_dataset.to_json("./intermediate/BASELINE_classified_frames_sample_all.jsonl", orient="records", lines=True)
underspecified_set.to_json("./intermediate/BASELINE_classified_frames_sample_UND.jsonl", orient="records", lines=True)
fully_specified_set.to_json("./intermediate/BASELINE_classified_frames_sample_FS.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 76.10ba/s]


1640588

## QA implementations

### Setup

In [2]:
underspecified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_frames_sample_UND.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

fully_specified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_frames_sample_FS.jsonl",
    split="all"
)

Generating train split: 440 examples [00:00, 29013.09 examples/s]
Generating train split: 384 examples [00:00, 36403.79 examples/s]


### Implementation

In [3]:
from openai import OpenAI
openai_api = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=openai_api)

In [4]:
from helper_functions_qa import (ask_short_answer, run_batch_shortQA_api, batch_QA_with_progress)

In [17]:
df_UND = pd.read_json('./intermediate/BASELINE_classified_frames_sample_UND.jsonl', lines=True)
df_FS = pd.read_json('./intermediate/BASELINE_classified_frames_sample_FS.jsonl', lines=True)

In [18]:
df_UND

,id,request,answer,qwen3_thinking,qwen3_model_response,qwen3_model_pred
0,2,How many years earlier would Punxsutawney Phil...,87,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""How many years earlier would ...",underspecified
1,3,"As of August 1, 2024, which country were holde...",France,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""As of August 1, 2024, which cou...",underspecified
2,4,What is the name of the vocalist from the firs...,Jens Kidman,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What is the name of the vocal...",underspecified
3,6,I have an element in mind and would like you t...,Mendelevium is named after Dmitri Mendeleev.,"<think>\nOkay, let me try to work through this...","{\n ""query"": ""I have an element in mind and w...",underspecified
4,7,"As of Aug 3, 2024, the artist who released the...",2,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""As of Aug 3, 2024, the artist...",underspecified
...,...,...,...,...,...,...
435,812,What is the name of the father of the first co...,Peter Treschow,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""What is the name of the fathe...",underspecified
436,814,Who was the winner of Tour de France the same ...,Louison Bobet,"<think>\nOkay, let's see. The user is asking f...","{\n ""query"": ""Who was the winner of Tour de F...",underspecified
437,815,A 2002 science fiction novel by an American au...,The Sea of Trolls trilogy,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""A 2002 science fiction novel ...",underspecified
438,822,Which movie musical produced a song that was i...,Fame,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""Which movie musical produced a ...",underspecified


In [19]:
df_FS

,id,request,answer,qwen3_thinking,qwen3_model_response,qwen3_model_pred
0,0,If my future wife has the same first name as t...,Jane Ballou,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""If my future wife has the sam...",fully specified
1,1,Imagine there is a building called Bronte towe...,37th,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""Imagine there is a building cal...",fully specified
2,5,"According to the 2000 United States census, wh...",506000,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""According to the 2000 United St...",fully specified
3,9,The Pope born Pietro Barbo ended a long-runnin...,The Battle of Hastings.,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""The Pope born Pietro Barbo ende...",fully specified
4,10,"An Australian artist, born the same year as ar...",Reve d'Or,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""An Australian artist, born th...",fully specified
...,...,...,...,...,...,...
379,817,I grew up in a village on Long Island in the T...,Sea Cliff Elementary School,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""I grew up in a village on Long ...",fully specified
380,818,Who was the Catholic Pope eleven years after E...,Eugene II,"<think>\nOkay, let's see. The user is asking, ...","{\n ""query"": ""Who was the Catholic Pope eleve...",fully specified
381,819,How many years after publishing his paper *On ...,Seven,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""How many years after publishing...",fully specified
382,820,This singer represented Sweden in Eurovision f...,Aquarius,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""This singer represented Sweden ...",fully specified


In [21]:
df_UND = df_UND.rename(columns={"request": "question"})
df_FS = df_FS.rename(columns={"request": "question"})
df_UND.to_json('./intermediate/BASELINE_classified_frames_sample_UND.jsonl', orient="records", lines=True)
df_FS.to_json('./intermediate/BASELINE_classified_frames_sample_FS.jsonl', orient="records", lines=True)

underspecified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_frames_sample_UND.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

fully_specified_set = load_dataset(
    "json",
data_files="./intermediate/BASELINE_classified_frames_sample_FS.jsonl",
    split="all"
)

Generating train split: 440 examples [00:00, 30444.15 examples/s]
Generating train split: 384 examples [00:00, 39299.53 examples/s]


In [22]:
short_results_UND = batch_QA_with_progress(
    underspecified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_short_answer",
    fill_value=["error"],
    client=client,
    model="gpt-4o-2024-11-20",
    temperature=0.0
)

Running model_short_answer: 100%|██████████| 44/44 [06:40<00:00,  9.09s/it]


In [23]:
# batch QA for FS
short_results_FS = batch_QA_with_progress(
    fully_specified_set,
    batch_fn=run_batch_shortQA_api,
    output_key="model_short_answer",
    fill_value=["error"],
    client=client,
    model="gpt-4o-2024-11-20",
    temperature=0.0
)

Running model_short_answer: 100%|██████████| 39/39 [05:54<00:00,  9.08s/it]


In [ ]:
qa_underspecified = deepcopy(underspecified_set)

for key in short_results_UND:
    qa_underspecified = qa_underspecified.add_column(key, short_results_UND[key])

qa_underspecified.to_json("./intermediate/BASELINE_frames_UND_qa_gpt.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 38.19ba/s]


1929001

In [ ]:
qa_fully_specified = deepcopy(fully_specified_set)

for key in short_results_FS:
    qa_fully_specified = qa_fully_specified.add_column(key, short_results_FS[key])

qa_fully_specified.to_json("./intermediate/BASELINE_frames_FS_qa_gpt.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 46.25ba/s]


1669525

In [ ]:
df = pd.read_json("./intermediate/BASELINE_frames_UND_qa_gpt.jsonl", lines=True)
df.to_csv('./output_csv/BASELINE_frames_UND_qa_gpt.csv')

df = pd.read_json("./intermediate/BASELINE_frames_FS_qa_gpt.jsonl", lines=True)
df.to_csv('./output_csv/BASELINE_frames_FS_qa_gpt.csv')

In [27]:
df

,id,question,answer,qwen3_thinking,qwen3_model_response,qwen3_model_pred,model_short_answer
0,0,If my future wife has the same first name as t...,Jane Ballou,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""If my future wife has the sam...",fully specified,[Elizabeth Parker]
1,1,Imagine there is a building called Bronte towe...,37th,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""Imagine there is a building cal...",fully specified,[The building would be 823 feet tall (Dewey De...
2,5,"According to the 2000 United States census, wh...",506000,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""According to the 2000 United St...",fully specified,"[200,000]"
3,9,The Pope born Pietro Barbo ended a long-runnin...,The Battle of Hastings.,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""The Pope born Pietro Barbo ende...",fully specified,[The War of the Roses]
4,10,"An Australian artist, born the same year as ar...",Reve d'Or,"<think>\nOkay, let me try to figure out if thi...","{\n ""query"": ""An Australian artist, born th...",fully specified,[The Lotuses]
...,...,...,...,...,...,...,...
379,817,I grew up in a village on Long Island in the T...,Sea Cliff Elementary School,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""I grew up in a village on Long ...",fully specified,[Southwoods Road Elementary School]
380,818,Who was the Catholic Pope eleven years after E...,Eugene II,"<think>\nOkay, let's see. The user is asking, ...","{\n ""query"": ""Who was the Catholic Pope eleve...",fully specified,[Pope Paschal I]
381,819,How many years after publishing his paper *On ...,Seven,"<think>\nOkay, let's tackle this query. The us...","{\n ""query"": ""How many years after publishing...",fully specified,[8 years]
382,820,This singer represented Sweden in Eurovision f...,Aquarius,"<think>\nOkay, let's tackle this query step by...","{\n ""query"": ""This singer represented Sweden ...",fully specified,[Aries]


## Evaluations

In [ ]:
underspecified_set_qa = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_frames_UND_qa_gpt.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

fully_specified_set_qa = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_frames_FS_qa_gpt.jsonl",
    split="all"
)

Generating train split: 440 examples [00:00, 33468.63 examples/s]
Generating train split: 384 examples [00:00, 34134.72 examples/s]


### Squad EM + F1

In [29]:
from helper_functions_qa import evaluate_squad_per_sample_multi_ref_pred

In [ ]:
# Official squad script for avg EM and F1, not possible for t-test


# Evaluate fully specified subset
dataset = load_dataset("json", data_files="./intermediate/BASELINE_frames_FS_qa_gpt.jsonl", split="train")

# 加载 HuggingFace 的 squad 评估器
squad_metric = evaluate.load("squad")

# 构造 predictions 和 references（标准格式）
predictions = [
    {
        "id": str(i),
        "prediction_text": pred[0] if isinstance(pred, list) and pred else ""
    }
    for i, pred in enumerate(dataset["model_short_answer"])
]

references = [
    {
        "id": str(i),
        "answers": {
            "text": ref if isinstance(ref, list) else [ref],
            "answer_start": [0] * len(ref if isinstance(ref, list) else [ref])
        }
    }
    for i, ref in enumerate(dataset["answer"])
]

# 计算 SQuAD-style EM 和 F1
results = squad_metric.compute(predictions=predictions, references=references)

# 打印平均指标
print(f"Exact Match: {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")

Exact Match: 18.49
F1 Score: 33.98


In [ ]:
# Official squad script for avg EM and F1, not possible for t-test


# Evaluate fully specified subset
dataset = load_dataset("json", data_files="./intermediate/BASELINE_frames_UND_qa_gpt.jsonl", split="train")

# 加载 HuggingFace 的 squad 评估器
squad_metric = evaluate.load("squad")

# 构造 predictions 和 references（标准格式）
predictions = [
    {
        "id": str(i),
        "prediction_text": pred[0] if isinstance(pred, list) and pred else ""
    }
    for i, pred in enumerate(dataset["model_short_answer"])
]

references = [
    {
        "id": str(i),
        "answers": {
            "text": ref if isinstance(ref, list) else [ref],
            "answer_start": [0] * len(ref if isinstance(ref, list) else [ref])
        }
    }
    for i, ref in enumerate(dataset["answer"])
]

# 计算 SQuAD-style EM 和 F1
results = squad_metric.compute(predictions=predictions, references=references)

# 打印平均指标
print(f"Exact Match: {results['exact_match']:.2f}")
print(f"F1 Score: {results['f1']:.2f}")

Exact Match: 12.95
F1 Score: 23.80


In [38]:
squad_scored_UND, UND_f1_list, UND_em_list = evaluate_squad_per_sample_multi_ref_pred(underspecified_set_qa, ref_col="answer")
squad_scored_UND.to_json("./intermediate/BASELINE_frames_UND_qa_gpt_with_squad_scores.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 46.52ba/s]


1936962

In [39]:
squad_scored_FS, FS_f1_list, FS_em_list = evaluate_squad_per_sample_multi_ref_pred(fully_specified_set_qa, ref_col="answer")
squad_scored_FS.to_json("./intermediate/BASELINE_frames_FS_qa_gpt_with_squad_scores.jsonl", orient="records", lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 54.12ba/s]


1676726

In [40]:
df = pd.read_json("./intermediate/BASELINE_frames_UND_qa_gpt_with_squad_scores.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_frames_UND_qa_gpt_with_squad_scores.csv')

df = pd.read_json("./intermediate/BASELINE_frames_FS_qa_gpt_with_squad_scores.jsonl", lines=True)
df.to_csv('./intermediate/BASELINE_frames_FS_qa_gpt_with_squad_scores.csv')

In [41]:
UND_mean_em = np.mean(UND_em_list)  # em_scores: EM list per sample
UND_mean_f1 = np.mean(UND_f1_list)  # f1_scores F1 list per sample
print(f"UND Exact Match (avg): {UND_mean_em * 100:.2f}")
print(f"UND F1 Score (avg): {UND_mean_f1 * 100:.2f}")

FS_mean_em = np.mean(FS_em_list)  # em_scores: EM list per sample
FS_mean_f1 = np.mean(FS_f1_list)  # f1_scores F1 list per sample
print(f"FS Exact Match (avg): {FS_mean_em * 100:.2f}")
print(f"FS F1 Score (avg): {FS_mean_f1 * 100:.2f}")

f1_tstat, f1_pval = ttest_ind(FS_f1_list, UND_f1_list, equal_var=False)
print(f"F1: t={f1_tstat:.3f}, p={f1_pval:.4f}")

em_tstat, em_pval = ttest_ind(FS_em_list, UND_em_list, equal_var=False)
print(f"EM: t={em_tstat:.3f}, p={em_pval:.4f}")

UND Exact Match (avg): 12.95
UND F1 Score (avg): 24.43
FS Exact Match (avg): 18.75
FS F1 Score (avg): 34.99
F1: t=4.107, p=0.0000
EM: t=2.265, p=0.0238


### Ragas

In [2]:
from helper_functions_qa import answer_accuracy

In [3]:
evaluator_llm = LangchainLLMWrapper(ChatDeepSeek(model="deepseek-chat", verbose=True, temperature=0))

UND_full = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_frames_UND_qa_gpt_with_squad_scores.jsonl",
    split="all"  # 必须指定 split，否则默认返回 DatasetDict
)

FS_full = load_dataset(
    "json",
    data_files="./intermediate/BASELINE_frames_FS_qa_gpt_with_squad_scores.jsonl",
    split="all"
)

In [ ]:
UND_ragas = await answer_accuracy(UND_full, evaluator_llm, ref_col = "answer")
UND_ragas.to_csv("./output_csv/frames_UND_gpt4o_Ragas.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 13.00ba/s]


1872649

In [ ]:
FS_ragas = await answer_accuracy(FS_full, evaluator_llm, ref_col = "answer")
FS_ragas.to_csv("./output_csv/frames_FS_gpt4o_Ragas.csv")

Creating CSV from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 15.85ba/s]


1619566

In [6]:
UND_ragas_AA = list(UND_ragas["ragas_AA_short"])
FS_ragas_AA = list(FS_ragas["ragas_AA_short"])

UND_mean_AA = np.mean(UND_ragas_AA)
print(f"UND AA (avg): {UND_mean_AA * 100:.2f}")


FS_mean_AA = np.mean(FS_ragas_AA)
print(f"FS AA (avg): {FS_mean_AA * 100:.2f}")

AA_tstat, AA_pval = ttest_ind(FS_ragas_AA, UND_ragas_AA, equal_var=False)
print(f"AA: t={AA_tstat:.3f}, p={AA_pval:.4f}")

UND AA (avg): 27.16
FS AA (avg): 44.92
AA: t=5.607, p=0.0000
